# Doublets_4 Clustering - Spheromer+ cells - Spleen 46 only

Publication source copy for the Doublets 2 and 4 sequencing analyses. Outputs and execution counts are cleared. See `../VALIDATION_REPORT.md` for the separate execution results and remaining limitations.

Run from the `analysis/` directory after supplying the bundled inputs. Final figure sections explicitly load preserved manuscript checkpoints; they are distinct from fresh clustering.

In [ ]:
from pathlib import Path
Path("../results/Doublets_4_Clustering_Sphr_HA_PRTN3_Sp46_larger_fonts").mkdir(parents=True, exist_ok=True)
path_revisionfigures = "../results/Doublets_4_Clustering_Sphr_HA_PRTN3_Sp46_larger_fonts/"

from threadpoolctl import threadpool_limits
threadpool_limits(limits=12, user_api="blas")


In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
from anndata import read_h5ad
from anndata import read_text
from collections import Counter
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from matplotlib.pyplot import rc_context
import anndata
import seaborn as sns
import regex as re
import os

In [ ]:
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=200, dpi_save=300)
sc.settings.n_jobs = 30
pd.options.display.max_columns = None
pd.options.display.max_rows = None

sc.logging.print_header()

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Import h5ad with processed data and BCR info

In [ ]:
path_data = '../data/Doublets/data/smartseq_doublets_4_analysis/'
adata = sc.read_h5ad(path_data+'Doublets4_7600cells_GeneSymbols_preprocessed_2xDoubletCounts.h5ad')

#the line below fixes an error that results when trying to run sc.tl.rank_genes_groups(adata,'leiden',method='wilcoxon')
#error is: KeyError: 'base'
adata.uns['log1p']["base"] = None
adata.obs.head()

# Filter interactions by imaging quality

In [ ]:
#filter by doublet score > 0
list_not_doubletscored = ['ArtDoub', 'Sing']
adata = adata[(adata.obs['sortedtype'].isin(list_not_doubletscored)) | (adata.obs['score']=='1')]
adata.shape

In [ ]:
Counter(adata.obs['sortedpop'])

# Keep only spheromers

In [ ]:
adata = adata[((adata.obs['sortedpop'].str.contains('-HA')) | (adata.obs['sortedpop'].str.contains('-PRTN3')))]

adata.shape

# Subset 1 donor

In [ ]:
adata = adata[adata.obs['SampleID']=='Spleen46']

adata.shape

In [ ]:
Counter(adata.obs['sortedpop'])

# Import index sort data

In [ ]:
#functions
def convert_well_to_colrow(well0):
    #row
    row0 = ord(well0[0])-64
    row0 = f"{row0:02d}"

    #col
    col0 = well0[1:3]

    #colrow
    colrow0 = 'c'+col0+'r'+row0

    return colrow0
convert_well_to_colrow('A12')

In [ ]:
path_idx_sort_data = '../data/Doublets/FACS/1_ImagingFACS/BD_S8/'
list_sort_dir = ['20240226_202004_20240226_kathuria_Sort_Sp46_HA_PRTN3_Spheromer_Doublets/',
                 '20240228_193700_20240228_kathuria_Sort_Sp46_HA_PRTN3_Spheromer_Doublets/',
                 '20240304_195443_20240304_kathuria_Sort_Sp46_HA_PRTN3_Spheromer_Doublets_Day10_1/',
                 '20240304_195626_20240304_kathuria_Sort_Sp46_HA_PRTN3_Spheromer_Doublets_Day10_2/',
                 '20240312_205116_20240312_kathuria_Sort_Sp55_HA_PRTN3_Spheromer_Doublets_Day3/',
                 '20240314_kathuria_Sort_Sp55_HA_PRTN3_Spheromer_Doublets_Day5/',
                 '20240314_kathuria_Sort_Sp55_HA_PRTN3_Spheromer_Doublets_Day5_2/',
                 '20240514_140535_20240319_kathuria_Sort_Sp55_HA_PRTN3_Spheromer_Doublets_Day10_1/',
                 '20240514_135626_20240319_kathuria_Sort_Sp55_HA_PRTN3_Spheromer_Doublets_Day10_2/']

list_df_plates = []
for sort_dir0 in list_sort_dir:
    for dir0 in os.listdir(path_idx_sort_data+sort_dir0):
        path_dir0 = path_idx_sort_data+sort_dir0+dir0+'/'
        if os.path.isdir(path_dir0) and 'Plate' in path_dir0:
            for filename0 in os.listdir(path_dir0):
                if filename0.endswith('.csv'):
                    df0 = pd.read_csv(path_dir0+filename0,header=12)
                    
                    df0 = df0[list(df0.columns[0:4])+list(df0.columns[23:24])+list(df0.columns[28:31])]

                    #convert well into column-row format
                    df0['ColumnRow'] = [convert_well_to_colrow(x) for x in list(df0['Well'])]

                    #convert filename into plate name
                    plate0 = 'plate0'
                    if 'Plate' in filename0.split('.csv')[0].split('_')[-1]:
                        plate0 = str(int(filename0.split('Plate')[-1].split('.')[0]))
                        
                    elif 'Plate' in filename0.split('_')[0]:
                        plate0 = str(int(filename0.split('_')[0].split('Plate')[-1]))

                    df0['96plate'] = plate0

                    #drop non-sorted wells
                    df0 = df0[df0['Event']==1]

                    #add day
                    if sort_dir0.rstrip('/') == '20240226_202004_20240226_kathuria_Sort_Sp46_HA_PRTN3_Spheromer_Doublets':
                        df0['Sort_Day'] = '3'
                    elif sort_dir0.rstrip('/') == '20240228_193700_20240228_kathuria_Sort_Sp46_HA_PRTN3_Spheromer_Doublets':
                        df0['Sort_Day'] = '5'
                    elif 'Day3' in path_dir0:
                        df0['Sort_Day'] = '3'
                    elif 'Day5' in path_dir0 or 'd5' in path_dir0:
                        df0['Sort_Day'] = '5'
                    elif 'Day10' in path_dir0:
                        df0['Sort_Day'] = '10'
                    else:
                        print(path_dir0)

                    #add to list
                    list_df_plates.append(df0)

df_plates = pd.concat(list_df_plates)
df_plates.sort_values(by=['96plate','ImagePath'],inplace=True)
df_plates = df_plates[df_plates['ImagePath']!='0']

#convert to log scale
df_plates['SPHR1 Alexa Fluor 488-A'] = [np.log10(x) for x in df_plates['SPHR1 Alexa Fluor 488-A']]
df_plates['SPHR2 APC-A'] = [np.log10(x) for x in df_plates['SPHR2 APC-A']]
df_plates['RBD-Sphr PE-A'] = [np.log10(x) for x in df_plates['RBD-Sphr PE-A']]
df_plates['CD19 BUV737-A'] = [np.log10(x) for x in df_plates['CD19 BUV737-A']]
print(df_plates.shape)
df_plates.head()

#add ColumnRow to match with index sort data
adata.obs['ColumnRow'] = [x[3:] for x in adata.obs['96plateloc']]

#merge
adata.obs = adata.obs.merge(df_plates, on=['ColumnRow','96plate'], how='left')
print(Counter(adata.obs['Sort Population']), Counter(adata.obs['sortedpop']))
print('Merge with index sort data',adata.obs.shape)

In [ ]:
#save dat metadata
adata.obs.to_csv(
    "../data/Doublets/data/smartseq_doublets_4_analysis/Doublets4_Spleen46_Smartseq_cell_metadata_inclDay.txt",
    sep="\t",
    index=True,
    
)

In [ ]:
df0.head()

# PCA

In [ ]:
#compute pca
sc.tl.pca(adata, svd_solver='arpack')

In [ ]:
sc.pl.pca_scatter(adata, color=['sortedpop'])#, save='_all_ages_droplet_facs_tissue.pdf')
sc.pl.pca_scatter(adata, color=['SampleID'])#, save='_all_ages_droplet_facs_tissue.pdf')
sc.pl.pca(adata, color=['384plate', 'sortedtype', 'sortedpop'], wspace=0.5)
sc.pl.pca_scatter(adata, color=['n_counts', 'n_genes'])

In [ ]:
sc.pl.pca_variance_ratio(adata, log=True)
sc.pl.pca_loadings(adata, include_lowest=True, components = '1,2,3,4,5,6')

# Cluster

In [ ]:
# Compute neighborhood graph followed by UMAP
sc.pp.neighbors(adata)
sc.tl.umap(adata)

#recommended method: Leiden graph clustering
sc.tl.leiden(adata,resolution=0.7)

In [ ]:
#plot
sc.pl.umap(adata, color=['leiden'], legend_fontsize=8)

sc.pl.umap(adata, color=['leiden','SampleID','Stimulation','sortedtype','sortedpop','Sort_Day'], legend_fontsize=8)
df_grouped = adata.obs.groupby('leiden')['sortedpop'].value_counts(normalize=True).unstack()
df_grouped.plot.bar(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

df_grouped = adata.obs.groupby('sortedtype')['sortedpop'].value_counts(normalize=True).unstack()
df_grouped.plot.bar(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

df_grouped = adata.obs.groupby('sortedpop')['leiden'].value_counts(normalize=True).unstack()
df_grouped.plot.bar(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

# Rename leiden clusters

In [ ]:
new_cluster_names = ['Naive B',
                     'T-B Cytotoxic',
                     'T-B Naive', #2
                     'B Unknown1',
                     'B Unknown2', #4
                     'T-B HA+ GC',
                     'T-B Plasmablast',
                     'T-B SOX5+ GC'
                     
]
adata.rename_categories('leiden', new_cluster_names)

adata

# Plots

In [ ]:
sc.pl.umap(adata, color=['leiden','sortedpop','sortedtype'], legend_fontsize=8)
df_grouped = adata.obs.groupby('leiden')['sortedpop'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

df_grouped = adata.obs.groupby('sortedpop')['leiden'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

df_grouped = adata.obs.groupby('leiden')['Stimulation'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

In [ ]:
df_grouped = adata.obs.groupby('Sort_Day')['leiden'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

df_grouped = adata.obs.groupby('leiden')['Sort_Day'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

df_grouped = adata.obs.groupby('SequencingRun')['Sort_Day'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

In [ ]:
adata.obs.head()

In [ ]:
adata_realdoub = adata[adata.obs['sortedtype']=='RealDoub']
df_grouped = adata_realdoub.obs.groupby('Sort_Day')['leiden'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

df_grouped = adata_realdoub.obs.groupby('leiden')['Sort_Day'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

df_grouped = adata_realdoub.obs.groupby('SequencingRun')['Sort_Day'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

In [ ]:
adata.obs.head()

In [ ]:
list_plot_genes = ['CD3E','CD4','CD8A','MS4A1','MKI67','TUBB', #general
                   'FOXP3','IL2RA','CXCR5','BCL6','TNFRSF18','PDCD1','LAG3','CTLA4','HAVCR2','PRF1','GZMA','GZMB','FASLG','GNLY','CD38','TNFRSF9','ITGAE','KLRG1','IL10','IL21','IFNG', #Treg
                   'RORC','CCR2','CCR5','CCR6','KLRB1','IL17RE', #Th17
                   'AICDA','SOX5','ZEB2','GSDME','GDA','MYC','CCND3','FAS','CD83','CXCR4','MIR155HG','CCL22','TNFSF4', 'EBI3','BCL2A1', 'FCRL3','FCRL2',#GC B cells
                   'PRDM1','EOMES',\
                    'CXCL9','CXCL10','CXCR3','CXCL13','CCL3','CCL4','CCL5','MPEG1','CCR7','SELL','CD22','BATF',
                   'IGHD','IGHM','IGHG1','IGHG2','IGHG3','IGHG4','IGHA1','IGHA2','JCHAIN','SIRPA','CD47','B3GAT1','IRF4'
                  ]

list_plot_genes = ['PRDM1','AICDA','ENPP2','LPAR2','LPAR5','LPAR6','LNCTAM34A','MIR34AHG','KCNQ5', 'TSHR']
list_plot_genes = ['CD3E','MS4A1','CXCR5','MIR155HG','CCL22','EBI3','CXCR3','CXCL9','CXCL10',#LZ
                   'MYC','CD40LG',
                    'CD83','CXCR4','BCL6','AICDA', #GC B cells
                   'SOX5','ITGAX','ZEB2','TBX21','FCER2','CD80','CD86','FCRL4','FCRL5','BHLHE40','HOPX','TFEC', #Atypical B cell markers
                   'IGHM','IGHG1','IGHG2','IGHG3','TCF7','PRDM1','IL21','TNFSF13B','TNFSF4','TNFRSF4','IL4','JCHAIN','MZB1','IL12B','IL12A','IL12RB2','SYK','IRF5','TNFRSF13B','ICOS','ICOSLG','STAT3','CD27','IGHM','IGHD','IL4R','TCL1A','FCER2','SELL',\
                   'CD1C','CD1D','CR2','TLR8','TLR7','TLR9','CD1C','CD70','CD27'
                  ]

sc.tl.dendrogram(adata, groupby='leiden')
with plt.rc_context():  # Use this to set figure params like size and dpi
    #scaled
    sc.pl.dotplot(adata, list_plot_genes, 'leiden', dendrogram=True, cmap='YlGnBu', standard_scale='var', colorbar_title='column scaled\nexpression',show=False)
    sc.pl.dotplot(adata, list_plot_genes, 'leiden', dendrogram=True, cmap='YlGnBu', standard_scale='none', colorbar_title='column non-scaled\nexpression',show=False)

In [ ]:
sc.pl.umap(adata, color=['leiden','AICDA','SOX5','CD4','CD8A','CXCL9','CXCL10','CXCR3'], legend_fontsize=8, cmap='YlGnBu')

In [ ]:
adata.obs.head()

In [ ]:
sc.pl.scatter(adata, x='CD4', y='CD8A', color='sortedpop', color_map='YlGnBu')
sc.pl.scatter(adata, x='CD4', y='PRF1', color='leiden', color_map='YlGnBu')
sc.pl.scatter(adata, x='CD8B', y='PRF1', color='leiden', color_map='YlGnBu')

sc.pl.scatter(adata, x='CD4', y='FOXP3', color='PRF1', color_map='YlGnBu')

In [ ]:
Counter(adata.obs['leiden'])

# Find DEGs

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='leiden', method='logreg', tie_correct=True, pts=True, key_added='logreg')
cluster_genes = pd.DataFrame(adata.uns['logreg']['names'])
cluster_genes.head(100)

In [ ]:
# # # Naive DEG analysis
sc.tl.rank_genes_groups(adata, 'sortedpop', groups=['RealDoub-HA'], reference='RealDoub-PRTN3', method='wilcoxon', key_added='wilcoxon')
cluster_genes = pd.DataFrame(adata.uns['wilcoxon']['names'])
cluster_genes.head(20)

In [ ]:
pd.DataFrame(adata.uns['wilcoxon']['logfoldchanges']).head(20)

# Merge BCR data with RNA data

In [ ]:
df_bcr = pd.read_csv(path_data+'Doublets4_TCR_BCR_productive_completeVDJ_UniqueClones.txt', sep='\t')
print('df_bcr ',df_bcr.shape)

### Drop nan IGH ###
df_bcr.dropna(subset=['locus_h','locus_l'],inplace=True)
print('Drop nan IGH ',df_bcr.shape)
df_bcr.reset_index(inplace=True, drop=True)

### Drop rows that have an 'N' in their sequence alignment ####
df_bcr = df_bcr[~(df_bcr['sequence_alignment_h'].str.contains('N'))]
df_bcr.reset_index(inplace=True, drop=True)
df_bcr = df_bcr[~(df_bcr['sequence_alignment_l'].str.contains('N'))]
df_bcr.reset_index(inplace=True, drop=True)
print('Drop rows that have an N in their sequence alignment', df_bcr.shape)
df_bcr.head()

#### Merge adata with bcr #####
adata.obs = adata.obs.merge(df_bcr, on=['barcode'], how='left')
adata.obs.head()

In [ ]:
df_grouped = adata.obs.groupby('leiden')['c_call_h'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

df_grouped = adata.obs.groupby('sortedpop')['c_call_h'].value_counts(normalize=True).unstack()
df_grouped.plot.barh(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

# Import TCR data and merge with RNA+BCR

In [ ]:
#import
path_data = '../data/Doublets/data/smartseq_doublets_4_analysis/'
df_ir = pd.read_csv(path_data+'Doublets4_TCR_BCR_productive_completeVDJ.txt', sep='\t')

#separate heavy and light chain
df_tcr_b = df_ir[df_ir['locus']=='TRB']
df_tcr_a = df_ir[df_ir['locus']=='TRA']

#keep only the first occurence of each TRB per donor
print(len(df_tcr_b))
df_tcr_b.sort_values(by=['v_score'],inplace=True,ascending=False)
df_tcr_b.drop_duplicates(subset=['cell_id'],inplace=True)
print(len(df_tcr_b))

#keep only the first occurence of each IgKL per donor
print(len(df_tcr_a))
df_tcr_a.sort_values(by=['v_score'],inplace=True,ascending=False)
df_tcr_a.drop_duplicates(subset=['cell_id'],inplace=True)
print(len(df_tcr_a))

#merge heavy and light chain
df_tcr = df_tcr_b.merge(df_tcr_a, on=['cell_id','barcode'], how='left', suffixes=('_b', '_a'))
print(df_tcr.shape)

### Drop nan TCR ###
df_tcr.dropna(subset=['locus_b','locus_a'],inplace=True)
print('Drop nan TCR ',df_tcr.shape)
df_tcr.reset_index(inplace=True, drop=True)

## Drop rows that have an 'N' in their sequence alignment ####

#drop columns that cause problem when saving adata (and is not useful)
df_tcr.dropna(axis=1, how='all',inplace=True)

#### Merge adata with bcr #####
adata.obs = adata.obs.merge(df_tcr, on=['barcode','cell_id'], how='left')
adata.obs.head()

# Add TCR and BCR expansion numbers

In [ ]:
###### Calculate clonal expansion by amino acid
adata.obs['junction_aa_h_count'] = [list(adata.obs['junction_aa_h']).count(x) if pd.notna(x) else np.nan for x in list(adata.obs['junction_aa_h'])]
adata.obs['junction_aa_h_expanded'] = ['Y' if x>1 else 'N' for x in list(adata.obs['junction_aa_h_count'])]

adata.obs['junction_aa_b_count'] = [list(adata.obs['junction_aa_b']).count(x) if pd.notna(x) else np.nan for x in list(adata.obs['junction_aa_b'])]
adata.obs['junction_aa_b_expanded'] = ['Y' if x>1 else 'N' for x in list(adata.obs['junction_aa_b_count'])]

###### Calculate clonal expansion by nucleotide
adata.obs['junction_h_count'] = [list(adata.obs['junction_h']).count(x) if pd.notna(x) else np.nan for x in list(adata.obs['junction_h'])]
adata.obs['junction_b_count'] = [list(adata.obs['junction_b']).count(x) if pd.notna(x) else np.nan for x in list(adata.obs['junction_b'])]

In [ ]:
sc.pl.umap(adata, color=['leiden','sortedpop','junction_aa_h_count','junction_aa_h_expanded','junction_aa_b_count','junction_aa_b_expanded'], legend_fontsize=8, cmap='YlGnBu')

# Save

In [ ]:
#save processed data
adata.write_h5ad(path_data+'Doublets_Sphr_SP46_1277cells_processed_2xDoubletCounts_TCR_BCR.h5ad') #include CD8

# Manuscript figures 20250303

In [ ]:
# Use the preserved manuscript object for the published figure layout.
path_data = '../data/Doublets/data/smartseq_doublets_4_analysis/'
adata = sc.read_h5ad('../data/reference_checkpoints/Doublets_Sphr_SP46_1277cells_processed_2xDoubletCounts_TCR_BCR.h5ad')

adata.uns['log1p']["base"] = None

#rename clusters

new_cluster_names = ['Naive B',
                     'cytotoxic T-B',
                     'T-Naive B',
                     'Memory B',
                     'Atypical B',
                     'T-GCB',
                     'T-Plasmablast',
                     'T-Atypical B'
]

adata.rename_categories('leiden', new_cluster_names)
Counter(adata.obs['leiden'])

#rename sortedpop categories

dict_sortedpop_old_to_new = {'RealDoub-HA': 'T-B HA+',
         'RealDoub-PRTN3': 'T-B PRTN3+', 
         'Sing-PRTN3': 'B PRTN3+', 
         'Sing-HA': 'B HA+'}

adata.obs['sortedpop'] = (
    adata.obs['sortedpop']
    .map(dict_sortedpop_old_to_new)
    .astype('category')
)
Counter(adata.obs['sortedpop']),Counter(adata.obs['leiden'])

# Merge Change-O BCR annotations

In [ ]:
# Import Change-O annotations for the current transcriptomic object.
#open changeo

df_changeo = pd.read_csv('../data/Doublets/data/smartseq_doublets_4_analysis/changeo/changeo_IGH_IGKL/allcells_outfmt_immcantation_db-pass_parse-select_clone-pass_germ-pass_Formatted_withSHM.tsv',sep='\t')

print('df_changeo',len(df_changeo))

#get barcode column
df_changeo['barcode'] = (
    df_changeo['sequence_id']
    .str.split(';').str[0]  # Take the part before the first semicolon
    .str.split('_').str[-2:]  # Take the last two elements split by underscore
    .str.join('-')  # Join them with a dash
)

#separate into IGH and IGKL
df_changeo_IGH = df_changeo[df_changeo['locus']=='IGH']
print('df_changeo_IGH',len(df_changeo_IGH))
df_changeo_IGKL = df_changeo[df_changeo['locus']!='IGH']
print('df_changeo_IGKL',len(df_changeo_IGKL))

#merge them with appropriate suffixes
df_changeo_merged = df_changeo_IGH.merge(df_changeo_IGKL, how='left', on=['barcode'],suffixes=['_changeo_h','_changeo_l'])
print('df_changeo_merged',len(df_changeo_merged))
df_changeo_merged.head()

#sort df_changeo_merged by support for V and J genes so highest scores are at top
df_changeo_merged.sort_values(by=['barcode','v_score_changeo_h','j_score_changeo_h','v_score_changeo_l','j_score_changeo_l'],inplace=True,ascending=False)

#drop duplicates of df_changeo_merged
df_changeo_merged.drop_duplicates(subset=['barcode'],inplace=True)
print('df_changeo_merged',len(df_changeo_merged))

#merge with adata
adata.obs = adata.obs.merge(df_changeo_merged, how='left', on=['barcode'])
print(adata.shape)

#### add clonal expansion data

#make clone
adata.obs['changeo_BCR_clone'] = adata.obs['junction_aa_changeo_h']+'-'+adata.obs['junction_aa_changeo_l']

#add counts
adata.obs['changeo_BCR_clone_count'] = adata.obs['changeo_BCR_clone'].map(adata.obs['changeo_BCR_clone'].value_counts())
adata.obs['clone_id_changeo_h_count'] = adata.obs['clone_id_changeo_h'].map(adata.obs['clone_id_changeo_h'].value_counts())
#filter

adata.obs.head()

## Figure 4

In [ ]:
path_figures = '../results/Doublets_4_Clustering_Sphr_HA_PRTN3_Sp46_larger_fonts/'

sc.settings.figdir = path_figures
sc.set_figure_params(dpi=300, dpi_save=600)

leiden_order_1 = ['T-Plasmablast',
                'T-Atypical B',
                'T-GCB',
                'cytotoxic T-B',
                'T-Naive B',
                'Atypical B',
                'Memory B',
                'Naive B',
               ]
               
leiden_order_2 = leiden_order_1[::-1]

sortedpop_order_1 = ['B HA+',
                     'T-B HA+',
                     'B PRTN3+',
                     'T-B PRTN3+']

sortedpop_order_2 = sortedpop_order_1[::-1]

leiden_palette_2 = [
    "#BDBDBD",  # Light Gray
    'dimgrey',
    "#969696",  # Medium Gray
    "#3182BD",  # Blue
    "#B2182B",  # Dark Red
    "#31A354",  # Green
    "#E6AB02",  # Goldenrod
    "#BC80BD",  # Purple
]

In [ ]:
Counter(adata.obs['leiden'])

In [ ]:
#######  Figure 4 ########

#all 
adata_plot_leiden = adata.copy()
adata_plot_leiden.obs['leiden'] = adata_plot_leiden.obs['leiden'].cat.set_categories(leiden_order_2, ordered=True)
sc.pl.umap(adata_plot_leiden, color='leiden', legend_fontsize=12, frameon=False, title='', palette=leiden_palette_2, save='_Fig_4_Sphr_SP46_leiden.pdf')

In [ ]:
## Plot distribution of sorted contacting populations
df_grouped = adata.obs.groupby('sortedpop')['leiden'].value_counts(normalize=True).unstack()
df_grouped = df_grouped.loc[['T-B HA+','T-B PRTN3+']]
df_grouped = df_grouped.reindex([sortedpop_order_1[3],sortedpop_order_1[1]])
df_grouped = df_grouped[leiden_order_2] #order columns

#singlet
df_grouped.plot.barh(stacked=True, color=leiden_palette_2, legend=False)

plt.grid(False)
plt.ylabel('')
sns.despine()
plt.savefig(path_figures+'barplot_Fig_4_Sphr_SP46_sortedpop_leiden.pdf', bbox_inches='tight')
plt.show()

In [ ]:
from scipy.stats import chi2_contingency, fisher_exact

# Get counts (not normalized) for the contingency table
df_counts = adata.obs.groupby('sortedpop')['leiden'].value_counts().unstack().fillna(0)
df_counts = df_counts.loc[['T-B HA+', 'T-B PRTN3+']]

# Remove columns with any zeros
df_counts_nonzero = df_counts.loc[:, (df_counts != 0).all(axis=0)]

# Chi-square test for overall distribution difference
chi2, p, dof, expected = chi2_contingency(df_counts_nonzero)
print(f"Chi-square test: χ² = {chi2:.2f}, p = {p:.2e}, dof = {dof}")

# Get counts
df_counts = adata.obs.groupby('sortedpop')['leiden'].value_counts().unstack().fillna(0)
df_counts = df_counts.loc[['T-B HA+', 'T-B PRTN3+']]

# Loop through all cell types
results = []
for cell_type in ['cytotoxic T-B','T-Naive B','T-GCB']:
    contingency_2x2 = pd.DataFrame({
        cell_type: df_counts[cell_type],
        'other': df_counts.drop(columns=cell_type).sum(axis=1)
    })
    odds_ratio, p_fisher = fisher_exact(contingency_2x2)
    
    # Calculate percentages for clarity
    pct_HA = df_counts.loc['T-B HA+', cell_type] / df_counts.loc['T-B HA+'].sum() * 100
    pct_PRTN3 = df_counts.loc['T-B PRTN3+', cell_type] / df_counts.loc['T-B PRTN3+'].sum() * 100
    
    results.append({
        'cell_type': cell_type,
        'HA+_pct': pct_HA,
        'PRTN3+_pct': pct_PRTN3,
        'OR (HA+ vs PRTN3+)': odds_ratio,
        'p_value': p_fisher
    })

df_results = pd.DataFrame(results)

# Multiple testing correction (optional but recommended)
from statsmodels.stats.multitest import multipletests
df_results['p_adj'] = multipletests(df_results['p_value'], method='fdr_bh')[1]

print(df_results.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
from statannotations.Annotator import Annotator

# Pairs you want to compare with an unpaired t-test
pairs = [
    ("B HA+", "B PRTN3+"),
    ("T-B HA+", "T-B PRTN3+"),
    ("B HA+", "T-B HA+"),
    ("B PRTN3+", "T-B PRTN3+")
]

violinplot_colors = ["#98C379","#264C22", "#E06C75",  "#8B0000"]

# Generate your violin plot but don't display it yet; capture the returned figure/axes
fig_or_axes = sc.pl.violin(
    adata, 
    keys=['clone_id_changeo_h_count'],  # same as list_plot_genes
    groupby='sortedpop',
    jitter=0.2,
    stripplot=True,
    multi_panel=True, 
    rotation=45,
    order=['B HA+','T-B HA+','B PRTN3+','T-B PRTN3+'],
    palette=violinplot_colors,
    ylabel=['IgH clonal expansion'],  # same as list_plot_titles
    show=False,                       # do not display yet
    return_fig=True                   # return the figure or axes
)
sns.despine()

# If multi_panel=True but you only have one gene, you typically get a single Axes object.
# Check whether what you get back is a single Axes or a list of Axes:
if isinstance(fig_or_axes, list):
    ax = fig_or_axes[0]  # use the first axis if you have only one panel
else:
    ax = fig_or_axes      # directly your Axes object if no list is returned

# Set up the statannotations Annotator
annotator = Annotator(
    ax,
    pairs,
    data=adata.obs,
    x='sortedpop',
    y='clone_id_changeo_h_count',
    order=['B HA+','T-B HA+','B PRTN3+','T-B PRTN3+'],
)

# Configure and apply the test (unpaired t-test)
annotator.configure(
    test='t-test_ind',   # unpaired t-test
    text_format='star',  # shows significance with asterisks
    loc='outside',
    verbose=2
)
annotator.apply_and_annotate()

# Now you can show or save your final figure
plt.tight_layout()
plt.savefig(path_figures + 'violinplot_Fig_4_sortedpop_clonalexpansion.pdf', bbox_inches='tight')
plt.show()

In [ ]:
list_plot_genes = ['CD3E','CD4','CD8A','MS4A1',#lineage
                   'FOXP3','IL2RA',#Treg
                   'PRF1','GZMA','GZMB','FASLG', #Cytotoxic
                   'BCL6','MYC','AICDA','BCL2A1', #GC
                   # ,'TNFSF4','SOX5','ITGAX','ZEB2',#GC/ ABC B cells

                  ]
with plt.rc_context():  # Use this to set figure params like size and dpi
    #scaled
    sc.pl.dotplot(adata, list_plot_genes, 'leiden', dendrogram=False, cmap='Reds', standard_scale='var', colorbar_title='column scaled\nexpression',show=True, swap_axes=False, categories_order=leiden_order_2, save='Fig_4_Sphr_SP46.pdf')

In [ ]:
## Plot isotype distribution of sorted populations
df_grouped = adata.obs.groupby('sortedpop')['c_call_h'].value_counts(normalize=True).unstack()
df_grouped = df_grouped.reindex(sortedpop_order_2) #order rows
df_grouped['IGHD*00'] = df_grouped['IGHD*00'] + df_grouped['IGHD*00,IGHM*00'] + df_grouped['IGHM*00,IGHD*00']
df_grouped = df_grouped[['IGHD*00','IGHM*00','IGHG1*00', 'IGHG2*00', 'IGHG3*00','IGHA1*00', 'IGHA2*00']] #order columns
df_grouped.columns = [x.split("*")[0] for x in df_grouped.columns]
isotype_palette = [
    "#A6CEE3",  # IGHD*00 (light blue)

    "#FF7F00",  # IGHM*00 (distinct orange)
    "#6A3D9A",  # IGHG3*00 (dark purple)
    "#9E83AB",  # IGHG2*00 (medium purple)
    "#CAB2D6",  # IGHG1*00 (light purple)

    "#0B3D0B","#4C9B4C"
]

#singlet
df_grouped.plot.barh(stacked=True, color=isotype_palette, legend=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.ylabel('')
sns.despine()
plt.savefig(path_figures+'barplot_Fig_4_Sphr_SP46_sortedpop_isotype.pdf', bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
from statannotations.Annotator import Annotator

# Pairs you want to compare with an unpaired t-test
pairs = [
    ("B HA+", "B PRTN3+"),
    ("T-B HA+", "T-B PRTN3+"),
    ("B HA+", "T-B HA+"),
    ("B PRTN3+", "T-B PRTN3+")
]

violinplot_colors = ["#98C379", "#264C22", "#E06C75", "#8B0000"]

# Generate your violin plot but don't display it yet; capture the returned figure/axes
fig_or_axes = sc.pl.violin(
    adata, 
    keys=['mu_freq_cdr_r_changeo_h'],  # same as list_plot_genes
    groupby='sortedpop',
    jitter=0.2,
    stripplot=True,
    multi_panel=True, 
    rotation=45,
    order=sortedpop_order_1,
    palette=violinplot_colors,
    ylabel=['IgH CDR SHM Frequency'],  # same as list_plot_titles
    show=False,                       # do not display yet
    return_fig=True                   # return the figure or axes
)
sns.despine()

# If multi_panel=True but you only have one gene, you typically get a single Axes object.
# Check whether what you get back is a single Axes or a list of Axes:
if isinstance(fig_or_axes, list):
    ax = fig_or_axes[0]  # use the first axis if you have only one panel
else:
    ax = fig_or_axes      # directly your Axes object if no list is returned

# Set up the statannotations Annotator
annotator = Annotator(
    ax,
    pairs,
    data=adata.obs,
    x='sortedpop',
    y='mu_freq_cdr_r_changeo_h',
    order=sortedpop_order_1
)

# Configure and apply the test (unpaired t-test)
annotator.configure(
    test='t-test_ind',   # unpaired t-test
    text_format='star',  # shows significance with asterisks
    loc='outside',
    verbose=2
)
annotator.apply_and_annotate()

# Now you can show or save your final figure
plt.tight_layout()
plt.savefig(path_figures + 'violinplot_Fig_4_sortedpop_shm.pdf', bbox_inches='tight')

plt.show()

# Figure S5C: fluorescence of index-sorted spheromer-positive events

In [ ]:
path_revisionfigures = '../results/Doublets_4_Clustering_Sphr_HA_PRTN3_Sp46_larger_fonts/'
sc.settings.figdir = path_revisionfigures
sc.set_figure_params(dpi=300, dpi_save=600)

## Antigen specific gating

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

groups = ['T-B HA+', 'B HA+', 'T-B PRTN3+', 'B PRTN3+']

for ax, sortpop0 in zip(axes, groups):
    adata_sub = adata[adata.obs['sortedpop'] == sortpop0]
    sc.pl.scatter(adata_sub, x='SPHR2 APC-A', y='SPHR1 Alexa Fluor 488-A', color='sortedpop', ax=ax, show=False, title=sortpop0,size=80)
    ax.title.set_fontsize(20)
    ax.xaxis.label.set_size(20)
    ax.yaxis.label.set_size(20)
    ax.tick_params(axis='both', labelsize=20)
    ax.set_xlim(1, 7)
    ax.set_ylim(1, 7)
    ax.grid(False)
    ax.get_legend().remove()
    ax.set_aspect('equal', adjustable='box')

plt.tight_layout()
plt.savefig(path_revisionfigures+'SP46_Spheromer_Gating.pdf', bbox_inches='tight')
plt.show()